# Notebook 01 - Images are numbers

*Time: about 45 minutes*

## What you will learn
- A picture on screen is really a **table of numbers**, and Python can read and change those numbers.
- How to load an image with `cv2.imread` and read its **shape** and **type**.
- The classic **BGR vs RGB** colour trap (why pictures sometimes look blue).
- How to read, change and **crop** pixels using simple slicing.

## Prerequisites
- You ran `python make_assets.py` once (it creates the pictures in the `assets/` folder).
- No Python experience needed. We go one line at a time. Press **Shift + Enter** to run a cell.

### What this code does

We tell Python which tools (libraries) we want to use today.
- `sys.path.append("..")` tells Python to also look one folder up, because that is where our helper file `utils.py` lives.
- `cv2` is OpenCV, the image library.
- `matplotlib.pyplot` (we call it `plt`) draws pictures in the notebook.
- `show` is our own tiny helper that draws a picture nicely.

In [ ]:
import sys                            # lets us change where Python looks for files
sys.path.append("..")                 # also look one folder up (utils.py lives there)
import cv2                            # OpenCV: reads and processes images
import matplotlib.pyplot as plt       # matplotlib: draws pictures and graphs
from utils import show                # our own helper that draws one picture
print("Libraries loaded - ready to go!")  # a visible sign that everything worked

### What you should see / Why we care

You should see `Libraries loaded - ready to go!`. If you see `ModuleNotFoundError`, you are probably using the wrong Python environment (see the README troubleshooting section).

### What this code does

We store the path of our picture in a variable called `IMAGE_PATH`, then ask OpenCV to read the file. Whatever we read becomes a variable called `img`.

**To use a different picture later, you only change the `IMAGE_PATH` line.**
Then we print three facts about `img`: what kind of thing it is, its shape, and what type of numbers it holds.

In [ ]:
IMAGE_PATH = "../assets/objects_shapes.png"   # <- the ONE line you change to use another picture
img = cv2.imread(IMAGE_PATH)                  # read the file into a NumPy array
print("type :", type(img))                    # what kind of object is it?
print("shape:", img.shape)                    # (height, width, channels)
print("dtype:", img.dtype)                    # what kind of numbers are inside?

### What you should see / Why we care

You should see `type: <class 'numpy.ndarray'>`, `shape: (420, 600, 3)` and `dtype: uint8`.

**Why we care:** an image is just a NumPy *array* (a grid of numbers). Everything we do in this course is arithmetic on that grid.

### What this code does

The shape `(420, 600, 3)` means **height = 420 rows**, **width = 600 columns**, **3 channels** (blue, green, red).
We unpack the three numbers into three named variables so the meaning is obvious. Then we ask for the smallest and largest number in the whole picture.

In [ ]:
height, width, channels = img.shape            # unpack the 3 numbers into 3 names
print("height  :", height, "pixels tall")      # number of rows
print("width   :", width, "pixels wide")       # number of columns
print("channels:", channels, "(blue, green, red)")  # numbers per pixel
print("smallest value:", img.min())            # darkest possible value is 0
print("largest value :", img.max())            # brightest possible value is 255

### What you should see / Why we care

`height: 420`, `width: 600`, `channels: 3`, smallest `0`, largest `255`.

**Why we care:** each pixel has 3 numbers between 0 (none of that colour) and 255 (full colour). `uint8` means "unsigned 8-bit integer": exactly the whole numbers 0 to 255.

## Showing the same image three ways

Showing a picture sounds easy, but there are traps. We show the **same** `img` three times.

### What this code does

**Way 1: `cv2.imshow`.** This opens a separate operating-system window. It often **freezes or shows nothing inside Jupyter**, so we keep it switched **off** with a variable called `TRY_IMSHOW`. If you want to try it on your own computer, change `False` to `True` (save your work first!).

In [ ]:
TRY_IMSHOW = False                        # change to True only if you want to experiment
if TRY_IMSHOW:                            # this block runs only when the switch is True
    cv2.imshow("my window", img)          # open a separate window
    cv2.waitKey(0)                        # wait until a key is pressed
    cv2.destroyAllWindows()               # close the window again
print("TRY_IMSHOW is", TRY_IMSHOW, "- in notebooks we use matplotlib instead")

### What you should see / Why we care

Nothing is drawn, on purpose. `cv2.imshow` is made for scripts, not notebooks. We use matplotlib from now on.

### What this code does

**Way 2: matplotlib, the careless way.** We hand `img` straight to `plt.imshow`. Watch the colours!

In [ ]:
plt.figure(figsize=(6, 4))                # make an empty figure 6 x 4 inches
plt.imshow(img)                           # draw the picture exactly as OpenCV gave it
plt.title("WRONG colours: red circles look blue")  # explain the problem in the title
plt.axis("off")                           # hide the axis numbers
plt.show()                                # display the figure

### What you should see / Why we care

The **red circles look blue** and the **blue triangles look red**.

**Why:** OpenCV stores each pixel as **B, G, R** (blue first). Matplotlib expects **R, G, B** (red first). We handed it the wrong order, so red and blue swapped. This is the most famous beginner gotcha in OpenCV.

### What this code does

**Way 3: the fix.** `cv2.cvtColor(img, cv2.COLOR_BGR2RGB)` reorders the three numbers of every pixel. We then show the converted copy. (Our helper `show` does this swap for us from now on.)

In [ ]:
rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)    # swap blue and red channels
plt.figure(figsize=(6, 4))                    # empty figure
plt.imshow(rgb)                               # draw the converted picture
plt.title("CORRECT colours: BGR converted to RGB")  # title
plt.axis("off")                               # hide the axis numbers
plt.show()                                    # display it

### What you should see / Why we care

Now the circles are **red**, the squares **green** and the triangles **blue**.

**Rule to remember:** OpenCV = BGR, matplotlib = RGB. Convert before you show.

### What this code does

Let us look at one pixel to *prove* the order. We read the pixel at row 90, column 100 (the middle of the first red circle) in both versions of the picture.

In [ ]:
print("OpenCV  (B, G, R):", img[90, 100])     # the original BGR pixel
print("matplot (R, G, B):", rgb[90, 100])     # the same pixel after conversion

### What you should see / Why we care

`[  0   0 220]` for OpenCV and `[220   0   0]` for matplotlib. Same colour, numbers in opposite order. A red pixel has **a lot of red (220)** and none of the others.

## Reading and changing pixels

In Python we write `img[row, column]`. Rows count **down** from the top, columns count **right** from the left. The top-left corner is `img[0, 0]`. This is different from maths class, where `y` goes up!

### What this code does

**Slicing** means taking a rectangle of pixels: `img[0:50, 0:100]` is "rows 0 up to (not including) 50, and columns 0 up to 100". It is the top-left corner. We print its shape and show it.

In [ ]:
corner = img[0:50, 0:100]                 # rows 0..49 and columns 0..99
print("corner shape:", corner.shape)      # 50 rows, 100 columns, 3 channels
show(corner, "img[0:50, 0:100] - the top-left corner", size=(4, 2))  # draw it

### What you should see / Why we care

`corner shape: (50, 100, 3)` and a small **white** rectangle (the top-left of the picture is empty background).

### What this code does

Now a **single pixel**. We pick the pixel at row 90, column 100, then unpack its three numbers into `blue`, `green` and `red`.

In [ ]:
pixel = img[90, 100]                      # one pixel = three numbers
blue, green, red = pixel                  # unpack them into three names
print("pixel (B, G, R):", pixel)          # the whole triple
print("blue =", blue, " green =", green, " red =", red)  # one at a time
print("a white background pixel:", img[5, 5])  # for comparison

### What you should see / Why we care

The circle pixel is `[0 0 220]` (strong red) and the background pixel is `[255 255 255]` (white = all three colours at full strength).

### What this code does

**Changing pixels.** First we make a **copy** with `img.copy()`, so the original stays safe. Changing one single pixel would be too small to see, so we paint a whole block: rows 10-40, columns 250-350 become yellow. Yellow in BGR is `(0, 255, 255)`: no blue, full green, full red.

In [ ]:
edited = img.copy()                       # work on a copy, keep the original safe
edited[10:40, 250:350] = (0, 255, 255)    # paint a yellow bar (B=0, G=255, R=255)
edited[100, 100] = (255, 0, 255)          # change ONE pixel (too small to notice!)
print("original pixel:", img[100, 100], " edited pixel:", edited[100, 100])  # compare
show(edited, "Edited copy: a yellow bar", size=(6, 4))  # draw the copy

### What you should see / Why we care

A **yellow bar** appears at the top. The printed pixel changed from `[0 0 220]` to `[255   0 255]`, but one pixel is invisible to the eye.

**Why we care:** drawing rectangles, masking, thresholding... everything later is "change numbers in a slice".

### What this code does

**Cropping** is just slicing, then keeping the result. The first red circle sits around row 90, column 100 with radius 40, so rows 50-130 and columns 60-140 cover it.

In [ ]:
crop = img[50:130, 60:140]                # rows 50..129, columns 60..139
print("crop shape:", crop.shape)          # 80 x 80 pixels, 3 channels
show(crop, "Cropped: the first red circle", size=(4, 4))  # draw the crop

### What you should see / Why we care

`crop shape: (80, 80, 3)` and a picture with only the first red circle.

**Why we care:** at the end of this course we will cut out letters, digits and objects automatically. The computer will find the numbers for the slice, but the slicing idea is exactly this one.

## Recap
| Idea | Code |
|---|---|
| Read a picture | `img = cv2.imread(IMAGE_PATH)` |
| Size | `img.shape` gives `(height, width, 3)` |
| Fix colours for matplotlib | `cv2.cvtColor(img, cv2.COLOR_BGR2RGB)` |
| One pixel | `img[row, column]` |
| A rectangle | `img[row1:row2, col1:col2]` |

**Next:** Notebook 02 looks at colour channels and grayscale.